In [37]:
import pandas as pd
import numpy as np

from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules
from scipy.sparse import csr_matrix

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
  

In [39]:
df=pd.read_csv("retail_cleaned.csv",low_memory=False)
df['InvoiceDate']=pd.to_datetime(df['InvoiceDate'])
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalAmount
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34


In [40]:
basket_df=df[['InvoiceNo', 'Description', 'Quantity','Country']].copy()

basket_df['Description'] = basket_df['Description'].astype(str).str.strip().str.upper()
basket_df=basket_df[basket_df['Quantity']>0]
baskey_df=basket_df.dropna(subset=['InvoiceNo','Description'])
basket_df.head()

,InvoiceNo,Description,Quantity,Country
0,536365,WHITE HANGING HEART T-LIGHT HOLDER,6,United Kingdom
1,536365,WHITE METAL LANTERN,6,United Kingdom
2,536365,CREAM CUPID HEARTS COAT HANGER,8,United Kingdom
3,536365,KNITTED UNION FLAG HOT WATER BOTTLE,6,United Kingdom
4,536365,RED WOOLLY HOTTIE WHITE HEART.,6,United Kingdom


In [41]:
basket_uk=basket_df[basket_df['Country']=='United Kingdom'].copy()
basket_uk.shape

(479985, 4)

In [42]:

invoice_item = (
    basket_uk
    .groupby(['InvoiceNo', 'Description'])['Quantity']
    .sum()
    .unstack()
    .fillna(0)
)

basket = invoice_item.apply(lambda col: np.where(col > 0, 1, 0), axis=0)
basket = pd.DataFrame(basket, index=invoice_item.index, columns=invoice_item.columns)


basket.shape


(18019, 3996)

In [43]:
basket = basket.astype(bool)


In [47]:
min_item_count = 500  
item_counts = basket.sum(axis=0)

frequent_items = item_counts[item_counts >= min_item_count].index
basket_small = basket[frequent_items]

basket_small.shape


(18019, 154)

In [51]:
basket_sparse = pd.DataFrame.sparse.from_spmatrix(
    csr_matrix(basket_small.values),
    index=basket_small.index,
    columns=basket_small.columns
)

basket_sparse.head()


C:\Users\adhir\AppData\Local\Temp\ipykernel_2412\1386851742.py:1: FutureWarning: Allowing arbitrary scalar fill_value in SparseDtype is deprecated. In a future version, the fill_value must be a valid value for the SparseDtype.subtype.
  basket_sparse = pd.DataFrame.sparse.from_spmatrix(


Description,6 RIBBONS RUSTIC CHARM,60 CAKE CASES VINTAGE CHRISTMAS,60 TEATIME FAIRY CAKE CASES,72 SWEETHEART FAIRY CAKE CASES,ALARM CLOCK BAKELIKE GREEN,ALARM CLOCK BAKELIKE IVORY,ALARM CLOCK BAKELIKE PINK,ALARM CLOCK BAKELIKE RED,ANTIQUE SILVER T-LIGHT GLASS,ASSORTED COLOUR BIRD ORNAMENT,...,VINTAGE SNAP CARDS,WHITE HANGING HEART T-LIGHT HOLDER,WOOD 2 DRAWER CABINET WHITE FINISH,WOOD BLACK BOARD ANT WHITE FINISH,WOODEN BOX OF DOMINOES,WOODEN FRAME ANTIQUE WHITE,WOODEN HEART CHRISTMAS SCANDINAVIAN,WOODEN PICTURE FRAME WHITE FINISH,WOODLAND CHARLOTTE BAG,ZINC METAL HEART DECORATION
InvoiceNo,,,,,,,,,,,,,,,,,,,,,
536365,0,0,0,0,0,0,0,0,0,0,...,0,True,0,0,0,0,0,0,0,0
536366,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536367,0,0,0,0,0,0,0,0,0,True,...,0,0,0,0,0,0,0,0,0,0
536368,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536369,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [53]:
frequent_itemsets = fpgrowth(
    basket_sparse,
    min_support=0.01,
    use_colnames=True
)

frequent_itemsets.head()


,support,itemsets
0,0.119984,(WHITE HANGING HEART T-LIGHT HOLDER)
1,0.076086,(ASSORTED COLOUR BIRD ORNAMENT)
2,0.041512,(HOME BUILDING BLOCK WORD)
3,0.033687,(LOVE BUILDING BLOCK WORD)
4,0.031189,(DOORMAT NEW ENGLAND)


In [55]:
rules = association_rules(
    frequent_itemsets,
    metric="lift",
    min_threshold=1.2
)

rules.sort_values("lift", ascending=False).head(10)


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
2761,"(WOODLAND CHARLOTTE BAG, RED RETROSPOT CHARLOT...","(CHARLOTTE BAG PINK POLKADOT, CHARLOTTE BAG SU...",0.023864,0.014429,0.010933,0.458140,31.750832,1.0,0.010589,1.818865,0.992182,0.399594,0.450206,0.607916
2760,"(CHARLOTTE BAG PINK POLKADOT, CHARLOTTE BAG SU...","(WOODLAND CHARLOTTE BAG, RED RETROSPOT CHARLOT...",0.014429,0.023864,0.010933,0.757692,31.750832,1.0,0.010589,4.028499,0.982684,0.399594,0.751769,0.607916
2753,"(WOODLAND CHARLOTTE BAG, RED RETROSPOT CHARLOT...","(CHARLOTTE BAG PINK POLKADOT, STRAWBERRY CHARL...",0.017482,0.020090,0.010933,0.625397,31.129904,1.0,0.010582,2.615862,0.985098,0.410417,0.617717,0.584798
2768,"(CHARLOTTE BAG PINK POLKADOT, STRAWBERRY CHARL...","(WOODLAND CHARLOTTE BAG, RED RETROSPOT CHARLOT...",0.020090,0.017482,0.010933,0.544199,31.129904,1.0,0.010582,2.155586,0.987720,0.410417,0.536089,0.584798
2758,"(RED RETROSPOT CHARLOTTE BAG, CHARLOTTE BAG SU...","(WOODLAND CHARLOTTE BAG, CHARLOTTE BAG PINK PO...",0.017870,0.019812,0.010933,0.611801,30.879682,1.0,0.010579,2.524963,0.985222,0.408714,0.603955,0.581811
2763,"(WOODLAND CHARLOTTE BAG, CHARLOTTE BAG PINK PO...","(RED RETROSPOT CHARLOTTE BAG, CHARLOTTE BAG SU...",0.019812,0.017870,0.010933,0.551821,30.879682,1.0,0.010579,2.191378,0.987175,0.408714,0.543666,0.581811
2759,"(CHARLOTTE BAG PINK POLKADOT, RED RETROSPOT CH...","(WOODLAND CHARLOTTE BAG, STRAWBERRY CHARLOTTE ...",0.017371,0.020756,0.010933,0.629393,30.323615,1.0,0.010572,2.642271,0.984117,0.402041,0.621538,0.578065
2762,"(WOODLAND CHARLOTTE BAG, STRAWBERRY CHARLOTTE ...","(CHARLOTTE BAG PINK POLKADOT, RED RETROSPOT CH...",0.020756,0.017371,0.010933,0.526738,30.323615,1.0,0.010572,2.076290,0.987519,0.402041,0.518372,0.578065
2770,"(CHARLOTTE BAG PINK POLKADOT, CHARLOTTE BAG SU...","(WOODLAND CHARLOTTE BAG, RED RETROSPOT CHARLOT...",0.021311,0.016927,0.010933,0.513021,30.308598,1.0,0.010572,2.018718,0.988063,0.400407,0.504636,0.579461
2751,"(WOODLAND CHARLOTTE BAG, RED RETROSPOT CHARLOT...","(CHARLOTTE BAG PINK POLKADOT, CHARLOTTE BAG SU...",0.016927,0.021311,0.010933,0.645902,30.308598,1.0,0.010572,2.763891,0.983656,0.400407,0.638191,0.579461


In [57]:
rules_clean = rules[[
    "antecedents",
    "consequents",
    "support",
    "confidence",
    "lift"
]].sort_values(["lift", "confidence"], ascending=False)

rules_clean.head(10)


,antecedents,consequents,support,confidence,lift
2760,"(CHARLOTTE BAG PINK POLKADOT, CHARLOTTE BAG SU...","(WOODLAND CHARLOTTE BAG, RED RETROSPOT CHARLOT...",0.010933,0.757692,31.750832
2761,"(WOODLAND CHARLOTTE BAG, RED RETROSPOT CHARLOT...","(CHARLOTTE BAG PINK POLKADOT, CHARLOTTE BAG SU...",0.010933,0.458140,31.750832
2753,"(WOODLAND CHARLOTTE BAG, RED RETROSPOT CHARLOT...","(CHARLOTTE BAG PINK POLKADOT, STRAWBERRY CHARL...",0.010933,0.625397,31.129904
2768,"(CHARLOTTE BAG PINK POLKADOT, STRAWBERRY CHARL...","(WOODLAND CHARLOTTE BAG, RED RETROSPOT CHARLOT...",0.010933,0.544199,31.129904
2758,"(RED RETROSPOT CHARLOTTE BAG, CHARLOTTE BAG SU...","(WOODLAND CHARLOTTE BAG, CHARLOTTE BAG PINK PO...",0.010933,0.611801,30.879682
2763,"(WOODLAND CHARLOTTE BAG, CHARLOTTE BAG PINK PO...","(RED RETROSPOT CHARLOTTE BAG, CHARLOTTE BAG SU...",0.010933,0.551821,30.879682
2759,"(CHARLOTTE BAG PINK POLKADOT, RED RETROSPOT CH...","(WOODLAND CHARLOTTE BAG, STRAWBERRY CHARLOTTE ...",0.010933,0.629393,30.323615
2762,"(WOODLAND CHARLOTTE BAG, STRAWBERRY CHARLOTTE ...","(CHARLOTTE BAG PINK POLKADOT, RED RETROSPOT CH...",0.010933,0.526738,30.323615
2751,"(WOODLAND CHARLOTTE BAG, RED RETROSPOT CHARLOT...","(CHARLOTTE BAG PINK POLKADOT, CHARLOTTE BAG SU...",0.010933,0.645902,30.308598
2770,"(CHARLOTTE BAG PINK POLKADOT, CHARLOTTE BAG SU...","(WOODLAND CHARLOTTE BAG, RED RETROSPOT CHARLOT...",0.010933,0.513021,30.308598


In [59]:
frequent_itemsets_ap = apriori(
    basket_small,
    min_support=0.01,
    use_colnames=True,
    max_len=2,        # VERY IMPORTANT
    low_memory=True
)

frequent_itemsets_ap.head()


,support,itemsets
0,0.047450,(6 RIBBONS RUSTIC CHARM)
1,0.032244,(60 CAKE CASES VINTAGE CHRISTMAS)
2,0.041789,(60 TEATIME FAIRY CAKE CASES)
3,0.030801,(72 SWEETHEART FAIRY CAKE CASES)
4,0.048615,(ALARM CLOCK BAKELIKE GREEN)


In [61]:
rules_clean.to_csv("association_rules.csv", index=False)